In [1]:
import pandas as pd
import numpy as np

mock_telecom = {
    'CustomerID':['1A','2B','3C','4D','5E','5E'],
    'Tenure_Months': [12,24,1,60,2,2],
    'Monthly_Charges': [50.5, 65.0, 20.0, 999.0, 30.0, 30.0],
    'Total_Charges': ['606.0', '1560.0', ' ', '6000.0', '60.0', '60.0'],
    'Contract_Type': [' Month-to-month ', 'One Year', 'month-to-month', 'Two year', 'One Year', 'One Year'],
    'Churn': ['Yes', 'No', 'Yes', 'No', 'Yes', 'Yes']
}

pd.DataFrame(mock_telecom).to_csv('Telecom_Customer_Churn.csv', index=False)

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
df = pd.read_csv('Telecom_Customer_Churn.csv')
print("Initial Data Info:\n")
df.info()

Initial Data Info:

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       6 non-null      str    
 1   Tenure_Months    6 non-null      int64  
 2   Monthly_Charges  6 non-null      float64
 3   Total_Charges    6 non-null      str    
 4   Contract_Type    6 non-null      str    
 5   Churn            6 non-null      str    
dtypes: float64(1), int64(1), str(4)
memory usage: 420.0 bytes


In [6]:
df['Total_Charges'] = pd.to_numeric(df['Total_Charges'], errors='coerce')
df['Total_Charges'] = df['Total_Charges'].fillna(df['Total_Charges'].mean())

df = df.drop_duplicates()

df['Contract_Type'] = df['Contract_Type'].str.strip().str.lower()

upper_limit = df['Monthly_Charges'].quantile(0.95)
df['Monthly_Charges'] = np.where(df['Monthly_Charges'] > upper_limit, upper_limit, df['Monthly_Charges'])

df['Avg_Charge_Per_Month'] = df['Total_Charges'] / df['Tenure_Months']

scaler = StandardScaler()

cols_to_scale = ['Tenure_Months', 'Monthly_Charges', 'Total_Charges', 'Avg_Charge_Per_Month']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

X = df.drop(['CustomerID', 'Churn'], axis=1) # Features (dropping ID because it doesn't predict anything)
y = df['Churn']                              # Target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining set size: {X_train.shape[0]} | Testing set size: {X_test.shape[0]}")

df.to_csv('Cleaned_Telecom_Churn.csv', index=False)
print("\nData perfectly cleaned and saved to 'Cleaned_Telecom_Churn.csv'!")


Training set size: 4 | Testing set size: 1

Data perfectly cleaned and saved to 'Cleaned_Telecom_Churn.csv'!


In [7]:
df_final = pd.read_csv('Cleaned_Telecom_Churn.csv')
df_final.head()

,CustomerID,Tenure_Months,Monthly_Charges,Total_Charges,Contract_Type,Churn,Avg_Charge_Per_Month
0,1A,-0.358660,-0.469799,-0.653201,month-to-month,1,-0.516706
1,2B,0.193124,-0.422832,-0.198557,one year,0,-0.494005
2,3C,-0.864461,-0.568591,-0.152234,month-to-month,1,1.998721
3,4D,1.848476,1.997422,1.917399,two year,0,-0.439210
4,5E,-0.818479,-0.536200,-0.913406,one year,1,-0.548801
